# Stage 09 — Homework Starter Notebook

In the lecture, we learned how to create engineered features. Now it’s your turn to apply those ideas to your own project data.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Example synthetic data (replace with your project dataset)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(
    "../../project/data/raw/market_returns_spy_spmo.csv"
)

df["date"] = pd.to_datetime(df["date"])

df.head()

## Engineered features — one is worked below; add at least two more

At least one of yours must encode a categorical column. `region` is the categorical
column here; the lecture shows three ways to encode it (one-hot, label, frequency).

In [ ]:
df["target_excess_next"] = df["excess_return"].shift(-1)

df["target_underperform_next"] = (
    df["target_excess_next"] < 0
).astype(int)

### Rationale for Feature 1
Explain why this feature may help a model. Reference your EDA.

In [ ]:
df["excess_lag1"] = df["excess_return"].shift(1)
print(
    df[["excess_lag1", "target_underperform_next"]]
    .corr()
)

df.groupby("target_underperform_next")["excess_lag1"].mean().plot(
    kind="bar"
)

plt.title("Previous Momentum Return by Next-Month Outcome")
plt.ylabel("Average Lagged Excess Return")
plt.show()

### Rationale for Feature 2
Explain why this feature may help a model. Reference your EDA.

In [ ]:
df["spy_vol_3m"] = df["spy_return"].rolling(3).std()

print(
    df[["spy_vol_3m", "target_underperform_next"]]
    .corr()
)

df.groupby("target_underperform_next")["spy_vol_3m"].mean().plot(
    kind="bar"
)

plt.title("Market Volatility by Next-Month Momentum Outcome")
plt.ylabel("Average 3-Month SPY Volatility")
plt.show()

### Rationale for Feature 3
Explain why this feature may help a model. Reference your EDA. If this is your
categorical encoding, say why you chose that encoding over the other two.

In [ ]:
df["market_regime"] = np.where(
    df["spy_return"] >= 0,
    "Up",
    "Down"
)

regime_dummies = pd.get_dummies(
    df["market_regime"],
    prefix="market_regime",
    dtype=int
)

df = pd.concat([df, regime_dummies], axis=1)

df.head()


print(
    df.groupby("market_regime")["target_underperform_next"].mean()
)

df.groupby("market_regime")["target_underperform_next"].mean().plot(
    kind="bar"
)

plt.title("Momentum Underperformance Rate by Market Regime")
plt.ylabel("Underperformance Rate")
plt.show()

In [ ]:
from src.features import add_features

df_test = pd.read_csv(
    "../../project/data/raw/market_returns_spy_spmo.csv"
)

df_test["date"] = pd.to_datetime(df_test["date"])

df_test = add_features(df_test)

df_test.head()